# Directional Wells Trayectories - Visualization

# Import Python Libraries

In [ ]:
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## Import Data

In [ ]:
Well_file = "Data/input/F-15/1.xml"
with open(Well_file) as file:
    data = file.read()

# Data Processing

In [ ]:
data_xml = BeautifulSoup(data, 'html.parser')
type(data_xml)

In [ ]:
float(data_xml.find_all("tvd")[1].text)

In [ ]:
float(data_xml.find_all("incl")[2].text)

In [ ]:
incl = data_xml.find_all("incl")
incl[3].text

In [ ]:
params = []
for tag in data_xml.find_all():
    params.append(str(tag.name))

len(params)

In [ ]:
params = set(params)
len(params)
params

In [ ]:
params = set([str(tag.name) for tag in data_xml.find_all()])
len(params)

In [ ]:
tvd = data_xml.find_all("tvd")
float(tvd[2].text)

In [ ]:
columns = ['azi', 'incl', 'md', 'tvd', 'dispns', 'dispew']
df = pd.DataFrame()
for col in columns:
    df[col] = [float(x.text) for x in data_xml.find_all(col)] 

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
wells = ['F-1 C',
        'F-4',
        'F-5',
        'F-7',
        'F-9',
        'F-11',
        'F-12',
        'F-14',
        'F-15']

preffix = "Data/input/"
suffix = "/1.xml"

df_all_wells = pd.DataFrame()
for well in wells:
    df = pd.DataFrame()
    WITSML_file = preffix + well + suffix
    with open(WITSML_file) as f:
        data = f.read()
    data_xml = BeautifulSoup(data, 'html.parser')
    for col in columns:
        df[col] = [float(x.text) for x in data_xml.find_all(col)]
    df['Well'] = well
    df_all_wells = pd.concat([df_all_wells, df], ignore_index=True)

In [ ]:
df_all_wells

In [ ]:
df_all_wells["Well"].unique()

In [ ]:
df_all_wells.shape

In [ ]:
df_all_wells.loc[df_all_wells['Well'] == 'F-12', ['azi', "incl", "md", "Well"]]

In [ ]:
df_group = df_all_wells.groupby('Well')['md'].max().reset_index()
df_group

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.bar(x=df_group["Well"], height=df_group["md"])

ax.set_xlabel("Wells")
ax.set_ylabel("MD (m)")

plt.show()

# Convert to negative all TVD Values

In [ ]:
df_all_wells['neg_tvd'] = df_all_wells['tvd']*-1

In [ ]:
fig = px.line_3d(df_all_wells, 'dispns', 'dispew', 'neg_tvd', color="Well")
fig.show()

# Plot well F-14

In [ ]:
df_all_wells[df_all_wells['Well']=='F-14']

In [ ]:
fig_14 = px.line_3d(df_all_wells[df_all_wells['Well']=='F-14'], 'dispns', 'dispew', 'neg_tvd')
fig_14.show()

In [ ]:
fig_5 = px.line_3d(df_all_wells[df_all_wells['Well']=='F-5'], 'dispns', 'dispew', 'neg_tvd')
fig_5.show()

# Output

Conver xml files to csv files

In [ ]:
wells = ['F-1 C',
        'F-4',
        'F-5',
        'F-7',
        'F-9',
        'F-11',
        'F-12',
        'F-14',
        'F-15']

preffix = "Data/output/"
suffix = ".csv"

for well in wells:
    df = df_all_wells[df_all_wells['Well']==well]
    df.to_csv(preffix + well + suffix, index=False)

# EDA (EXploratory Data Analysis)

In [ ]:
df_all_wells

In [ ]:
corr = df_all_wells[["azi", "incl", "md", "tvd", "dispns", "dispew", "neg_tvd"]].corr()

fig, ax1 = plt.subplots(figsize=(20,8))
sns.heatmap(data=corr, cmap='RdYlGn', annot=True, linewidths=0.01, linecolor='black', square=True, ax=ax1)
plt.show()

## Plots for Categorical Variables

In [ ]:
sns.boxplot(data=df_all_wells, x='Well', y='md')

In [ ]:
sns.barplot(data=df_all_wells, x='Well', y='tvd')

In [ ]:
columns = ['azi', 'incl', 'md', 'tvd', 'dispns', 'dispew']
fig, ax = plt.subplots(len(columns), 1, figsize=(8,20))

for col, axes in zip(columns, ax):
    sns.boxplot(data=df_all_wells, x='Well', y=col, ax=axes)
    plt.tight_layout()